In [11]:
import os
import dotenv
from langchain.chains.conversation.base import ConversationChain
from langchain.chains.llm import LLMChain
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm=ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=40
)

一、接口的最底层 ChatMessageHistory

In [3]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()
history.add_user_message('你好')
history.add_ai_message('python很有趣')
history.add_user_message('计算1+1=')

print(history.messages)

res=llm.invoke(history.messages)
print(res.content)


[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='python很有趣', additional_kwargs={}, response_metadata={}), HumanMessage(content='计算1+1=', additional_kwargs={}, response_metadata={})]
1 + 1 = 2。


二、ConversationBufferMemory

In [4]:
#举例1 以字符串形式返回
from langchain.memory import ConversationBufferMemory

memroy=ConversationBufferMemory()

#inputs对应用户消息 outputs对应ai消息
memroy.save_context(inputs={'input':'你好 我叫小明'},outputs={'output':'很高兴认识你'})
memroy.save_context(inputs={'input':'帮我回答一下1+2等于几'},outputs={'output':'3'})

#返回字典的结构的key叫history
print(memroy.load_memory_variables({}))

{'history': 'Human: 你好 我叫小明\nAI: 很高兴认识你\nHuman: 帮我回答一下1+2等于几\nAI: 3'}


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7812\3505536237.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memroy=ConversationBufferMemory()


In [5]:
#举例2 以消息列表形式返回
from langchain.memory import ConversationBufferMemory

memroy=ConversationBufferMemory(return_messages=True)

memroy.save_context(inputs={'input':'你好 我叫小明'},outputs={'output':'很高兴认识你'})
memroy.save_context(inputs={'input':'帮我回答一下1+2等于几'},outputs={'output':'3'})

print(memroy.load_memory_variables({}))
print('\n')
print(memroy.chat_memory.messages)


{'history': [HumanMessage(content='你好 我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我回答一下1+2等于几', additional_kwargs={}, response_metadata={}), AIMessage(content='3', additional_kwargs={}, response_metadata={})]}


[HumanMessage(content='你好 我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我回答一下1+2等于几', additional_kwargs={}, response_metadata={}), AIMessage(content='3', additional_kwargs={}, response_metadata={})]


In [6]:
#举例3 结合llm,PromptTemplate

#提示词模板
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {history}
人类问题: {question}
回复:
"""
)
#memory
memory = ConversationBufferMemory()

#Chain 会使用memory中的history给提示词模板的history赋值
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

res = chain.invoke(input={'question': '你好 我叫小明'})

print(res)
print('\n')
res=chain.invoke(input={'question': '我的名字是什么'})
print(res)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7812\257808188.py:15: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)


{'question': '你好 我叫小明', 'history': '', 'text': '你好，小明！很高兴认识你！有什么我可以帮助你的吗？'}


{'question': '我的名字是什么', 'history': 'Human: 你好 我叫小明\nAI: 你好，小明！很高兴认识你！有什么我可以帮助你的吗？', 'text': '你的名字是小明。请问你有什么想聊的或者需要帮助的呢？'}


In [7]:
#举例4 基于举例3 修改memory_key

#提示词模板
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {chat}
人类问题: {question}
回复:
"""
)
#memory 修改memory_key为chat
memory = ConversationBufferMemory(memory_key='chat')


chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

res = chain.invoke(input={'question': '你好 我叫小明'})

print(res)
print('\n')
res=chain.invoke(input={'question': '我的名字是什么'})
print(res)

{'question': '你好 我叫小明', 'chat': '', 'text': '你好，小明！很高兴认识你。请问有什么我可以帮助你的吗？'}


{'question': '我的名字是什么', 'chat': 'Human: 你好 我叫小明\nAI: 你好，小明！很高兴认识你。请问有什么我可以帮助你的吗？', 'text': '你的名字是小明。请问你想聊些什么呢？'}


In [8]:
#举例5 结合ChatPromptTemplate

from langchain_core.messages import SystemMessage
from langchain_core.prompts import MessagesPlaceholder,ChatPromptTemplate,HumanMessagePromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个与人类对话的机器人。"),
    MessagesPlaceholder(variable_name='history'),
    ("human","问题：{question}")
])


memory = ConversationBufferMemory(return_messages=True)

llm_chain = LLMChain(prompt=prompt,llm=llm, memory=memory)

#第一次调用就把问题和答复写入了history
res1 = llm_chain.invoke({"question": "中国首都在哪里？"})
print(res1,end="\n\n")


{'question': '中国首都在哪里？', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都在北京。', additional_kwargs={}, response_metadata={})], 'text': '中国的首都在北京。'}



In [9]:
res2 = llm_chain.invoke({"question": "我刚刚问了什么"})
print(res2)

{'question': '我刚刚问了什么', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都在北京。', additional_kwargs={}, response_metadata={}), HumanMessage(content='我刚刚问了什么', additional_kwargs={}, response_metadata={}), AIMessage(content='你刚刚问了“中国首都在哪里？”这个问题。', additional_kwargs={}, response_metadata={})], 'text': '你刚刚问了“中国首都在哪里？”这个问题。'}


三、ConversationChain的使用    将memory和Chain结合为一步 甚至可以结合提还差模板

In [20]:
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {history}
人类问题: {input}
回复:
"""
)
# #memory
# memory = ConversationBufferMemory()
#
# #Chain 会使用memory中的history给提示词模板的history赋值
# chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

chain=ConversationChain(llm=llm, prompt=prompt_template)

res = chain.invoke(input={'input': '你好 我叫小明'})

print(res)

{'input': '你好 我叫小明', 'history': '', 'response': '你好，小明！很高兴认识你。有什么我可以帮助你的吗？'}


In [21]:
res=chain.invoke(input={'input': '我的名字是什么'})
print(res)

{'input': '我的名字是什么', 'history': 'Human: 你好 我叫小明\nAI: 你好，小明！很高兴认识你。有什么我可以帮助你的吗？', 'response': '你的名字是小明。你还有其他想分享的或者问我的问题吗？'}


In [22]:
#内部也有默认的提示词模板 一个变量是input 另一个是history
chain=ConversationChain(llm=llm)
res = chain.invoke(input={'input': '今天星期三'})
print(res)
res=chain.invoke(input={'input':'明天星期几'})
print(res)

{'input': '今天星期三', 'history': '', 'response': '是的，今天是星期三！星期三通常被认为是一周的中间，这一天的名字在中文中表示“三”。你今天有什么特别的计划吗？或者有想聊'}
{'input': '明天星期几', 'history': 'Human: 今天星期三\nAI: 是的，今天是星期三！星期三通常被认为是一周的中间，这一天的名字在中文中表示“三”。你今天有什么特别的计划吗？或者有想聊', 'response': '明天是星期四！星期四在一周中排在第三位，通常被视为接近周末的日子。你对周四有什么期待吗？或者有什么特别'}


四、ConversationBufferWindowMemory

In [24]:
from langchain.memory import ConversationBufferWindowMemory

# 1. 初始化消息存储（新版必须显式指定，用于存放历史消息）
chat_history = ChatMessageHistory()

# 2. 初始化窗口记忆（k=2 表示只保留最近2条消息）
memory = ConversationBufferWindowMemory(
    k=2,
    chat_memory=chat_history,  # 绑定消息存储
    return_messages=False  # 设为 False，返回文本格式的 history（而非消息对象列表）
)

chat_history.add_message(HumanMessage(content="你好 我叫小明"))
chat_history.add_message(AIMessage(content="很高兴认识你"))
chat_history.add_message(HumanMessage(content="帮我回答一下1+2等于几"))
chat_history.add_message(AIMessage(content="3"))
chat_history.add_message(HumanMessage(content="一周有几天"))
chat_history.add_message(AIMessage(content="7"))

print(memory.load_memory_variables({}))

{'history': 'Human: 帮我回答一下1+2等于几\nAI: 3\nHuman: 一周有几天\nAI: 7'}


In [26]:
#结合llm
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {history}
人类问题: {input}
回复:
"""
)
#memory 只保留前一条消息
memory = ConversationBufferWindowMemory(k=1)

#Chain 会使用memory中的history给提示词模板的history赋值
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

res = chain.invoke(input={'input': '你好 我叫小明'})
print(res)
res=chain.invoke(input={'input':'今天星期三'})
print(res)
res=chain.invoke(input={'input':'明天星期几'})
print(res)
res=chain.invoke(input={'input':'我叫什么名字'})
print(res)

{'input': '你好 我叫小明', 'history': '', 'text': '你好，小明！很高兴认识你。你今天过得怎么样？'}
{'input': '今天星期三', 'history': 'Human: 你好 我叫小明\nAI: 你好，小明！很高兴认识你。你今天过得怎么样？', 'text': 'AI: 星期三过得怎么样？有什么特别的事情发生吗？'}
{'input': '明天星期几', 'history': 'Human: 今天星期三\nAI: AI: 星期三过得怎么样？有什么特别的事情发生吗？', 'text': '明天是星期四。你有什么计划吗？'}
{'input': '我叫什么名字', 'history': 'Human: 明天星期几\nAI: 明天是星期四。你有什么计划吗？', 'text': '我不知道你的名字，不过你可以告诉我你的名字吗？'}
